In [9]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from statsmodels.tsa.stattools import coint
import warnings

# Suppress technical warnings for cleaner output
warnings.filterwarnings('ignore')

def get_stooq_data(ticker, start, end):
    symbol = f"{ticker}.US" if "." not in ticker else ticker
    try:
        df = web.DataReader(symbol, 'stooq', start, end)
        if df is None or df.empty: return None
        return df['Close'].sort_index()
    except:
        return None

def run_multi_window_test(pairs_list):
    # Define the 7-year windows
    windows = [
        ("2004-01-01", "2011-01-01", "Great Financial Crisis"),
        ("2013-01-01", "2020-01-01", "COVID-19 / Mid-Caps"),
        ("2019-01-01", "2026-01-01", "Modern Regime (2022 Bear/Current)")
    ]
    
    all_results = []

    for start, end, label in windows:
        print(f"\n>>> TESTING WINDOW: {label} ({start} to {end})")
        
        for s1, s2 in pairs_list:
            y = get_stooq_data(s1, start, end)
            x = get_stooq_data(s2, start, end)
            
            if y is None or x is None:
                continue
            
            # Align data
            combined = pd.concat([y, x], axis=1).dropna()
            combined.columns = ['S1', 'S2']
            
            if len(combined) < 250: # Ensure at least 1 year of trading days
                continue
            
            # Cointegration Test
            score, p_value, _ = coint(combined['S1'], combined['S2'])
            
            all_results.append({
                'Window': label,
                'Pair': f"{s1}-{s2}",
                'P-Value': round(p_value, 4),
                'Significant': "YES" if p_value < 0.1 else "no"
            })

    # Display as a summary table
    df = pd.DataFrame(all_results)
    pd.set_option('display.max_rows', None)
    print("\n" + "="*60)
    print("STRESS TEST SUMMARY: COINTEGRATION ACROSS ERAS")
    print("="*60)
    print(df.to_string(index=False))

# Your Ticker List
trading_pairs = [
    ("AAP", "IBM"), ("EWA", "EWC"), ("NKE", "AES"), ("MRK", "VZ"),
    ("GS", "CVX"), ("MRK", "DD"), ("AES", "XOM"), ("CVX", "AMG"),
    ("VZ", "KO"), ("VZ", "JNJ"), ("VZ", "PG"), ("KO", "WMT"),
    ("WMT", "V"), ("GE", "IBM"), ("WMT", "XOM"), ("TRV", "XOM"),
    ("PG", "DD"), ("NKE", "VZ"), ("CSCO", "XOM"), ("ACN", "VZ")
]

if __name__ == "__main__":
    run_multi_window_test(trading_pairs)


>>> TESTING WINDOW: Great Financial Crisis (2004-01-01 to 2011-01-01)

>>> TESTING WINDOW: COVID-19 / Mid-Caps (2013-01-01 to 2020-01-01)

>>> TESTING WINDOW: Modern Regime (2022 Bear/Current) (2019-01-01 to 2026-01-01)

STRESS TEST SUMMARY: COINTEGRATION ACROSS ERAS
                           Window     Pair  P-Value Significant
           Great Financial Crisis  AAP-IBM   0.7451          no
           Great Financial Crisis  EWA-EWC   0.3089          no
           Great Financial Crisis  NKE-AES   0.9706          no
           Great Financial Crisis   MRK-VZ   0.1730          no
           Great Financial Crisis   GS-CVX   0.2682          no
           Great Financial Crisis   MRK-DD   0.5167          no
           Great Financial Crisis  AES-XOM   0.8494          no
           Great Financial Crisis  CVX-AMG   0.7886          no
           Great Financial Crisis    VZ-KO   0.4325          no
           Great Financial Crisis   VZ-JNJ   0.3886          no
           Great Financial 